# D1 cascade detector — governed Colab launcher

This notebook launches only the one-class detector baseline on the approved private train/validation bundle. It has no test route. The runner pins the approved bundle, dataset manifest, builders, detector lock and `yolo11s.pt`, and writes text, SQLite and JSON execution logs.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import re
import subprocess
from pathlib import Path

REPOSITORY = 'https://github.com/ScottPhantom/Traffic_SIgn_Detection.git'
BRANCH = 'DEV/Rroject_Pineline'
REPO = Path('/content/Traffic_Sign_Detector')
DRIVE_ROOT = Path('/content/drive/MyDrive/CPV391/M1_D1_Colab')
BUNDLE = DRIVE_ROOT / 'm1_colab_train_val.tar.gz'
STATE = DRIVE_ROOT / 'cascade_runs' / os.environ.get('D1C_RUN_ID', 'smoke_seed20261018')
SMOKE = True
SEED = 20261018
assert SEED in (20261018, 20261019, 20261020)
assert BUNDLE.is_file(), BUNDLE
assert not STATE.exists(), (
    f'Refusing non-empty/reused state path: {STATE}'
)
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY, str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
assert re.fullmatch(r'[0-9a-f]{40}', commit) and not dirty
print({'commit': commit, 'bundle': str(BUNDLE), 'state': str(STATE), 'smoke': SMOKE})

In [ ]:
subprocess.run(['pip', 'install', 'uv==0.9.5'], check=True)
cmd = [
    'python', str(REPO / 'detector/colab/run_d1_cascade_colab.py'), 'run',
    '--repo-root', str(REPO), '--bundle', str(BUNDLE), '--state-dir', str(STATE),
    '--seed', str(SEED), '--epochs', '1' if SMOKE else '100', '--imgsz', '1280',
    '--batch', '8', '--device', '0', '--workers', '4', '--fraction', '1.0',
    '--patience', '30', '--tag', 'smoke' if SMOKE else 'candidate',
    '--download-approved-weight',
]
subprocess.run(cmd, cwd=REPO, check=True)

In [ ]:
import json
import sqlite3

execution = json.loads((STATE / 'colab_execution.json').read_text())
assert execution['status'] == 'SUCCEEDED' and execution['sealed_test_accessed'] is False
with sqlite3.connect(f"file:{STATE / 'DB_logger.db'}?mode=ro", uri=True) as db:
    assert db.execute('PRAGMA integrity_check').fetchone() == ('ok',)
    print('log rows:', db.execute('SELECT COUNT(*) FROM logs').fetchone()[0])
print(execution)